# Earthquake Risk Prediction using Machine Learning

This notebook builds a **risk classification system** using historical earthquake data from the **USGS Earthquake Catalog API**.

> Important: this project does **not** claim to predict the exact time or occurrence of earthquakes. It estimates a **risk level** (`Low`, `Medium`, `High`) based on patterns in historical records.

## Project goals
- Collect real earthquake data from the USGS API
- Clean and explore the dataset
- Engineer useful time-based features
- Train machine learning models to classify earthquake risk
- Evaluate model performance with standard metrics
- Make a sample risk prediction for a new input


## 1. Import Libraries

In [ ]:
# Core libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Display settings
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")

## 2. Data Collection

We will fetch earthquake data directly from the **USGS Earthquake Catalog API** using a CSV response. The query below collects records from January 1, 2010 onward.

Reference: https://earthquake.usgs.gov/fdsnws/event/1/

In [ ]:
# Build the USGS API query URL.
# We request CSV so pandas can read it directly.
base_url = "https://earthquake.usgs.gov/fdsnws/event/1/query.csv"

query_params = {
    "format": "csv",
    "starttime": "2010-01-01",
    "orderby": "time-asc",
    "minmagnitude": 0,
}

query_string = "&".join([f"{key}={value}" for key, value in query_params.items()])
data_url = f"{base_url}?{query_string}"

print("USGS data URL:")
print(data_url)

# Load the dataset directly from the API.
df = pd.read_csv(data_url)

print(f"Dataset shape: {df.shape}")
df.head()

## 3. Data Preprocessing

In this section we:
- keep only relevant columns
- handle missing values
- convert the `time` column to datetime
- engineer time-based features
- create the target risk level from magnitude

### Risk categories
- `Low`: magnitude < 3
- `Medium`: 3 <= magnitude < 5
- `High`: magnitude >= 5

Note: `magnitude` is used to create the target label, so we will **not** include it in the training features. This avoids **target leakage**.

In [ ]:
# Select the most important columns for this project.
columns_to_keep = ["time", "latitude", "longitude", "depth", "mag"]
earthquakes = df[columns_to_keep].copy()

# Rename mag to magnitude for clarity.
earthquakes.rename(columns={"mag": "magnitude"}, inplace=True)

# Check missing values.
print("Missing values before cleaning:")
print(earthquakes.isnull().sum())

# Drop rows with missing values in required columns.
earthquakes.dropna(inplace=True)

# Convert time column to datetime.
earthquakes["time"] = pd.to_datetime(earthquakes["time"], errors="coerce")
earthquakes.dropna(subset=["time"], inplace=True)

# Create useful time-based features.
earthquakes["year"] = earthquakes["time"].dt.year
earthquakes["month"] = earthquakes["time"].dt.month
earthquakes["day"] = earthquakes["time"].dt.day
earthquakes["hour"] = earthquakes["time"].dt.hour
earthquakes["day_of_week"] = earthquakes["time"].dt.dayofweek

# Create risk labels from magnitude.
def magnitude_to_risk(magnitude):
    if magnitude < 3:
        return "Low"
    elif magnitude < 5:
        return "Medium"
    else:
        return "High"

earthquakes["risk_level"] = earthquakes["magnitude"].apply(magnitude_to_risk)

print(f"Cleaned dataset shape: {earthquakes.shape}")
earthquakes.head()

In [ ]:
# Inspect the class distribution.
risk_counts = earthquakes["risk_level"].value_counts().sort_index()
print("Risk level counts:")
print(risk_counts)

plt.figure(figsize=(8, 5))
sns.countplot(data=earthquakes, x="risk_level", order=["Low", "Medium", "High"])
plt.title("Distribution of Earthquake Risk Levels")
plt.xlabel("Risk Level")
plt.ylabel("Count")
plt.show()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Magnitude distribution
plt.figure(figsize=(10, 6))
sns.histplot(earthquakes["magnitude"], bins=40, kde=True, color="teal")
plt.title("Magnitude Distribution")
plt.xlabel("Magnitude")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Earthquake frequency over time by month
monthly_counts = (
    earthquakes.set_index("time")
    .resample("M")
    .size()
    .reset_index(name="earthquake_count")
)

plt.figure(figsize=(14, 6))
plt.plot(monthly_counts["time"], monthly_counts["earthquake_count"], color="darkorange")
plt.title("Earthquake Frequency Over Time")
plt.xlabel("Time")
plt.ylabel("Number of Earthquakes")
plt.show()

In [ ]:
# Depth vs magnitude scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=earthquakes.sample(min(10000, len(earthquakes)), random_state=42),
    x="depth",
    y="magnitude",
    hue="risk_level",
    alpha=0.7,
)
plt.title("Depth vs Magnitude")
plt.xlabel("Depth")
plt.ylabel("Magnitude")
plt.show()

In [ ]:
# Correlation heatmap for numeric features
numeric_cols = ["latitude", "longitude", "depth", "magnitude", "year", "month", "day", "hour", "day_of_week"]
corr_matrix = earthquakes[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 5. Feature Selection

We use location, depth, and engineered time features as model inputs.

### Selected features
- `latitude`
- `longitude`
- `depth`
- `year`
- `month`
- `day`
- `hour`
- `day_of_week`

We intentionally exclude:
- `time` because raw datetime values are not directly usable by most models
- `magnitude` because the target is derived from it

In [ ]:
# Define features (X) and target (y)
feature_columns = ["latitude", "longitude", "depth", "year", "month", "day", "hour", "day_of_week"]
X = earthquakes[feature_columns].copy()
y = earthquakes["risk_level"].copy()

# Train-test split with stratification to preserve class balance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
X_train.head()

## 6. Model Building

We will train the following models:
- Logistic Regression
- Random Forest
- XGBoost (optional, if installed)

In [ ]:
# Define machine learning models.
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                multi_class="auto",
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

# Try to include XGBoost if the package is available.
xgb_available = False
label_encoder = LabelEncoder()

try:
    from xgboost import XGBClassifier

    xgb_available = True
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_test_encoded = label_encoder.transform(y_test)

    models["XGBoost"] = XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        random_state=42,
    )
    print("XGBoost is available and will be trained.")
except ImportError:
    print("XGBoost is not installed. The notebook will continue with Logistic Regression and Random Forest.")

In [ ]:
# Helper function to train and evaluate each model.
results = []
trained_models = {}

for model_name, model in models.items():
    print(f"\nTraining: {model_name}")

    if model_name == "XGBoost" and xgb_available:
        model.fit(X_train, y_train_encoded)
        y_pred_encoded = model.predict(X_test)
        y_pred = label_encoder.inverse_transform(y_pred_encoded)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    trained_models[model_name] = model

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
    })

    print(classification_report(y_test, y_pred, zero_division=0))

results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False).reset_index(drop=True)
results_df

## 7. Model Evaluation

In [ ]:
# Compare model performance visually.
results_melted = results_df.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(12, 6))
sns.barplot(data=results_melted, x="Metric", y="Score", hue="Model")
plt.title("Model Performance Comparison")
plt.ylim(0, 1)
plt.show()

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print(f"Best model based on weighted F1 score: {best_model_name}")

In [ ]:
# Confusion matrix for the best model.
if best_model_name == "XGBoost" and xgb_available:
    best_pred = label_encoder.inverse_transform(best_model.predict(X_test))
else:
    best_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, best_pred, labels=["Low", "Medium", "High"])

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Low", "Medium", "High"],
    yticklabels=["Low", "Medium", "High"],
)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

## 8. Prediction

We now create a sample input and ask the best model to classify the earthquake risk level.

In [ ]:
# Create a sample input.
# These values can be changed to test different scenarios.
sample_input = pd.DataFrame([
    {
        "latitude": 34.05,
        "longitude": -118.25,
        "depth": 12.5,
        "year": 2025,
        "month": 7,
        "day": 15,
        "hour": 14,
        "day_of_week": 1,
    }
])

if best_model_name == "XGBoost" and xgb_available:
    predicted_class_encoded = best_model.predict(sample_input)[0]
    predicted_risk = label_encoder.inverse_transform([predicted_class_encoded])[0]
    probabilities = best_model.predict_proba(sample_input)[0]
    class_names = label_encoder.classes_
else:
    predicted_risk = best_model.predict(sample_input)[0]
    probabilities = best_model.predict_proba(sample_input)[0]
    class_names = best_model.classes_

print("Sample input:")
display(sample_input)
print(f"Predicted earthquake risk level: {predicted_risk}")

probability_df = pd.DataFrame({
    "Risk Level": class_names,
    "Probability": probabilities,
}).sort_values(by="Probability", ascending=False)

probability_df

## 9. Visualization

The chart below shows the predicted probability for each risk class.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=probability_df, x="Risk Level", y="Probability", palette="mako")
plt.title(f"Predicted Risk Probabilities - {best_model_name}")
plt.ylim(0, 1)
plt.show()

## 10. Conclusion

### Key insights
- Historical earthquake data can be used to build a **risk classification** pipeline.
- Location, depth, and time-derived features may help identify patterns associated with different magnitude-based risk levels.
- Tree-based models such as Random Forest or XGBoost often perform well for non-linear relationships.

### Limitations
- This notebook does **not** predict the exact occurrence of future earthquakes.
- The target is derived from historical magnitude ranges, so the model is best understood as a **risk categorization tool**.
- Important scientific factors such as tectonic plate boundaries, fault line data, and seismic wave measurements are not included here.
- Earthquake datasets are often imbalanced, which can make rare high-risk events harder to classify.

### Next improvements
- Add geospatial features such as distance to known fault zones
- Use time-window aggregation features
- Try hyperparameter tuning and cross-validation
- Build an interactive dashboard for live risk scoring